# 🎮 SQL 탐정단 — e스포츠 결승전의 사라진 1위 기록
### 김부장 × 혼이 · 학교 e스포츠 기록 조작 의혹 추적

> **학생용 / Google Colab 실행 가능 / 15개 자동점검 미션 + CRUD 실습**

학교 e스포츠 결승전에서 서준의 **9,800점이 갑자기 6,800점으로 바뀌었습니다.** 누가, 언제, 어떤 계정으로 기록을 수정했을까요? SQL로 단서를 모아 사건을 해결하세요.

**주의:** 이 데이터와 사건은 수업용 창작물입니다. 로그에 표시된 계정과 실제 행동한 사람은 같다고 단정할 수 없습니다.

## 🗺️ 사건 해결 로드맵
| 단계 | 학습 내용 | 사건 역할 |
|---|---|---|
| 0 | DB·테이블·행·열 이해 | 증거 보관함 읽기 |
| 1 | SELECT, WHERE, LIKE, ORDER BY, LIMIT | 용의선상 좁히기 |
| 2 | COUNT, GROUP BY, HAVING | 팀별 기록 분석 |
| 3 | PK, FK 개념, INNER/LEFT JOIN, NULL | 분리된 장부 연결 |
| 4 | 다중 JOIN, 시간 조건 | 점수 변경과 접속 로그 대조 |
| 5 | CREATE, INSERT, UPDATE, DELETE | 안전한 연습용 기록 관리 |

**권장 수업:** 2차시 × 50분 또는 프로젝트형 100~110분. `Shift+Enter`로 한 셀씩 실행합니다.

## 0. 사건 데이터 — 표를 왜 나눌까요?
- `players`: 사람 정보 (`player_id`가 각 선수의 고유번호)
- `scores`: 현재 점수 (`player_id`로 선수를 참조)
- `score_history`: 점수 변경 전후 이력
- `access_logs`: 누가 어떤 계정으로 어떤 작업을 했는지 기록한 로그

점수 장부에 선수 이름·팀·학년을 매번 복사하지 않고 **고유번호로 연결**하면 수정·중복 오류를 줄일 수 있습니다. `players.player_id`는 **기본키(PK)**, 다른 장부의 `player_id`는 **연결 열**로 사용합니다. 이 실습에서는 설명을 위해 외래키 제약을 강제하지 않습니다.

## 1. 준비 셀 — 아래 셀을 가장 먼저 실행하세요 (수정하지 않기)

In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display

conn = sqlite3.connect(':memory:')
conn.executescript("""
DROP TABLE IF EXISTS access_logs;
DROP TABLE IF EXISTS score_history;
DROP TABLE IF EXISTS scores;
DROP TABLE IF EXISTS players;
CREATE TABLE players (player_id INTEGER PRIMARY KEY, name TEXT NOT NULL, team TEXT NOT NULL, grade INTEGER NOT NULL);
CREATE TABLE scores (score_id INTEGER PRIMARY KEY, player_id INTEGER UNIQUE NOT NULL, points INTEGER NOT NULL, updated_at TEXT NOT NULL);
CREATE TABLE score_history (history_id INTEGER PRIMARY KEY, player_id INTEGER NOT NULL, old_points INTEGER NOT NULL, new_points INTEGER NOT NULL, changed_at TEXT NOT NULL);
CREATE TABLE access_logs (log_id INTEGER PRIMARY KEY, actor_id INTEGER NOT NULL, target_id INTEGER NOT NULL, action TEXT NOT NULL, log_time TEXT NOT NULL, device TEXT NOT NULL);
""")
conn.executemany('INSERT INTO players VALUES (?,?,?,?)', [(101, '서준', '블루', 3), (102, '민아', '레드', 3), (103, '도윤', '레드', 3), (104, '하린', '블루', 2), (105, '지후', '그린', 2), (106, '소율', '그린', 3), (107, '유나', '레드', 2)])
conn.executemany('INSERT INTO scores VALUES (?,?,?,?)', [(1, 101, 6800, '14:32'), (2, 102, 8700, '14:08'), (3, 103, 8200, '14:10'), (4, 104, 7300, '14:05'), (5, 105, 7600, '14:12'), (6, 106, 7000, '14:07')])
conn.executemany('INSERT INTO score_history VALUES (?,?,?,?,?)', [(1, 101, 9100, 9400, '14:01'), (2, 101, 9400, 9800, '14:15'), (3, 102, 8500, 8700, '14:08'), (4, 103, 7900, 8200, '14:10'), (5, 101, 9800, 6800, '14:32')])
conn.executemany('INSERT INTO access_logs VALUES (?,?,?,?,?,?)', [(1, 102, 101, 'VIEW', '14:20', 'PC-01'), (2, 103, 101, 'LOGIN', '14:28', 'PC-07'), (3, 103, 101, 'UPDATE_SCORE', '14:32', 'PC-07'), (4, 104, 101, 'VIEW', '14:33', 'PC-03'), (5, 101, 101, 'LOGIN', '14:40', 'PC-02'), (6, 103, 103, 'VIEW', '14:43', 'PC-07'), (7, 106, 102, 'VIEW', '14:50', 'PC-05')])
conn.commit()
print('✅ 사건 데이터베이스 준비 완료 · players 7명 / scores 6건 / score_history 5건 / access_logs 7건')

In [ ]:
def run_sql(query):
    """학생 SQL을 화면에 출력. TODO가 있으면 힌트만 표시."""
    if 'TODO' in query.upper() or '-- 작성' in query:
        print('🟡 TODO 부분을 실제 SQL로 바꿔 보세요.')
        return None
    try:
        frame = pd.read_sql_query(query, conn)
        display(frame)
        print(f'✅ 검색 완료: {len(frame)}행')
        return frame
    except Exception as e:
        print('🔍 SQL 오류:', e)
        return None

RESULTS = {}
ANSWERS = {}
def check(mission, student_query):
    if 'TODO' in student_query.upper() or '-- 작성' in student_query:
        RESULTS[mission] = False
        print('🟡 미완료: TODO를 수정하세요.'); return
    try:
        actual = conn.execute(student_query).fetchall()
        expected = conn.execute(ANSWERS[mission]).fetchall()
        good = sorted(map(repr,actual)) == sorted(map(repr,expected)) and len(actual)==len(expected)
        RESULTS[mission] = good
        print(('🏆 정답입니다!' if good else '🔎 결과가 달라요. 조건·연결·정렬을 다시 확인하세요.') + f' (결과 {len(actual)}행)')
    except Exception as e:
        RESULTS[mission] = False
        print('⚠️ 실행 오류:',e)

def progress():
    solved = sum(RESULTS.get(i,False) for i in ANSWERS)
    print(f'🎮 해결한 미션: {solved}/{len(ANSWERS)}')
    for i in ANSWERS: print(('✅' if RESULTS.get(i) else '⬜'),f'미션 {i:02d}')

## 2. 증거 열람 — 먼저 테이블 구조와 데이터를 보세요

In [ ]:
for table in ['players','scores','score_history','access_logs']:
    print('\n📂',table)
    display(pd.read_sql_query(f'SELECT * FROM {table};',conn))

### SQL 문장 지도
```sql
SELECT name          -- 볼 열
FROM players         -- 읽을 테이블
WHERE team = '레드'  -- 조건
ORDER BY name;       -- 정렬
```
**기본 실행 순서:** `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT` (개념적 순서). `;`는 문장 끝입니다.

### 🕵️ 미션 01 · 사건 01 | 명단 확보
**목표:** SELECT · 모든 선수의 이름과 팀을 확인하세요.

**힌트:** `SELECT name, team`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_01 = '''
SELECT TODO
FROM players
ORDER BY player_id;
'''
run_sql(mission_01)

In [ ]:
ANSWERS[1] = 'SELECT name, team FROM players ORDER BY player_id;'
check(1, mission_01)

### 🕵️ 미션 02 · 사건 02 | 레드팀 수색
**목표:** WHERE · 레드팀 선수만 조회하세요.

**힌트:** `team = '레드'`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_02 = '''
SELECT name, team FROM players
WHERE TODO
ORDER BY player_id;
'''
run_sql(mission_02)

In [ ]:
ANSWERS[2] = "SELECT name, team FROM players WHERE team='레드' ORDER BY player_id;"
check(2, mission_02)

### 🕵️ 미션 03 · 사건 03 | 이름의 단서
**목표:** LIKE · 이름이 ‘ㅅ’으로 시작하는 선수는? (한글 글자 기준)

**힌트:** `name LIKE '서%' OR name LIKE '소%'`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_03 = '''
SELECT name FROM players
WHERE TODO
ORDER BY player_id;
'''
run_sql(mission_03)

In [ ]:
ANSWERS[3] = "SELECT name FROM players WHERE name LIKE '서%' OR name LIKE '소%' ORDER BY player_id;"
check(3, mission_03)

### 🕵️ 미션 04 · 사건 04 | 고득점자 찾기
**목표:** WHERE + ORDER BY · 현재 점수 7,500점 이상 순위표

**힌트:** `points >= 7500 / points DESC`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_04 = '''
SELECT player_id, points FROM scores
WHERE TODO
ORDER BY TODO;
'''
run_sql(mission_04)

In [ ]:
ANSWERS[4] = 'SELECT player_id, points FROM scores WHERE points >= 7500 ORDER BY points DESC;'
check(4, mission_04)

### 🕵️ 미션 05 · 사건 05 | TOP 3
**목표:** LIMIT · 현재 점수 1~3위의 선수번호와 점수

**힌트:** `3`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_05 = '''
SELECT player_id, points FROM scores
ORDER BY points DESC
LIMIT TODO;
'''
run_sql(mission_05)

In [ ]:
ANSWERS[5] = 'SELECT player_id, points FROM scores ORDER BY points DESC LIMIT 3;'
check(5, mission_05)

#### 🧠 잠깐 정리
- `WHERE`: 행을 거릅니다.
- `ORDER BY points DESC`: 큰 점수부터 정렬합니다.
- `LIMIT 3`: 정렬된 결과에서 처음 세 행만 남깁니다.
- `LIKE '서%'`: '서'로 시작하는 문자열입니다.

### 🕵️ 미션 06 · 사건 06 | 선수 수와 점수 수
**목표:** COUNT · 선수 전체 인원 집계

**힌트:** `COUNT(*)`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_06 = '''
SELECT TODO AS 선수수 FROM players;
'''
run_sql(mission_06)

In [ ]:
ANSWERS[6] = 'SELECT COUNT(*) AS 선수수 FROM players;'
check(6, mission_06)

### 🕵️ 미션 07 · 사건 07 | 팀별 인원
**목표:** GROUP BY · 팀별 선수 인원을 출력하세요.

**힌트:** `team`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_07 = '''
SELECT team, COUNT(*) AS 인원
FROM players
GROUP BY TODO
ORDER BY team;
'''
run_sql(mission_07)

In [ ]:
ANSWERS[7] = 'SELECT team, COUNT(*) AS 인원 FROM players GROUP BY team ORDER BY team;'
check(7, mission_07)

### 🕵️ 미션 08 · 사건 08 | 두 명 이상인 팀
**목표:** HAVING · 팀원이 2명 이상인 팀만 출력

**힌트:** `COUNT(*) >= 2`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_08 = '''
SELECT team, COUNT(*) AS 인원
FROM players
GROUP BY team
HAVING TODO
ORDER BY team;
'''
run_sql(mission_08)

In [ ]:
ANSWERS[8] = 'SELECT team, COUNT(*) AS 인원 FROM players GROUP BY team HAVING COUNT(*) >= 2 ORDER BY team;'
check(8, mission_08)

#### 🧠 WHERE와 HAVING은 달라요
- `WHERE`: 묶기 **전** 개별 행을 거릅니다.
- `GROUP BY`: 같은 팀끼리 묶습니다.
- `HAVING`: 묶은 **후** 팀별 인원 같은 조건을 판단합니다.

### 🕵️ 미션 09 · 사건 09 | 두 장부 합치기
**목표:** INNER JOIN · 선수 이름과 현재 점수를 연결하세요.

**힌트:** `P.player_id = S.player_id`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_09 = '''
SELECT P.name, S.points
FROM players AS P
INNER JOIN scores AS S ON TODO
ORDER BY S.points DESC;
'''
run_sql(mission_09)

In [ ]:
ANSWERS[9] = 'SELECT P.name, S.points FROM players P INNER JOIN scores S ON P.player_id=S.player_id ORDER BY S.points DESC;'
check(9, mission_09)

### 🕵️ 미션 10 · 사건 10 | 기록 없는 선수
**목표:** LEFT JOIN + IS NULL · 점수가 등록되지 않은 선수를 찾으세요.

**힌트:** `S.score_id IS NULL`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_10 = '''
SELECT P.name
FROM players P
LEFT JOIN scores S ON P.player_id=S.player_id
WHERE TODO;
'''
run_sql(mission_10)

In [ ]:
ANSWERS[10] = 'SELECT P.name FROM players P LEFT JOIN scores S ON P.player_id=S.player_id WHERE S.score_id IS NULL;'
check(10, mission_10)

#### 🧠 INNER JOIN / LEFT JOIN
`INNER JOIN`은 두 장부에서 연결되는 행만, `LEFT JOIN`은 **왼쪽 장부의 모든 행**을 남깁니다. 연결 상대가 없으면 오른쪽 열은 `NULL`입니다. `NULL`은 0이나 빈 글자가 아니므로 `= NULL`이 아닌 **`IS NULL`** 로 확인하세요.

### 🕵️ 미션 11 · 사건 11 | 최초 1위 찾기
**목표:** 점수 변경 내역에서 9,800점 기록의 선수번호를 확인하세요.

**힌트:** `new_points = 9800`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_11 = '''
SELECT player_id, old_points, new_points, changed_at
FROM score_history
WHERE TODO;
'''
run_sql(mission_11)

In [ ]:
ANSWERS[11] = 'SELECT player_id,old_points,new_points,changed_at FROM score_history WHERE new_points=9800;'
check(11, mission_11)

### 🕵️ 미션 12 · 사건 12 | 수상한 변경
**목표:** 점수가 떨어진 변경 내역만 확인하세요.

**힌트:** `new_points < old_points`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_12 = '''
SELECT player_id, old_points, new_points, changed_at
FROM score_history
WHERE TODO;
'''
run_sql(mission_12)

In [ ]:
ANSWERS[12] = 'SELECT player_id,old_points,new_points,changed_at FROM score_history WHERE new_points < old_points;'
check(12, mission_12)

### 🕵️ 미션 13 · 사건 13 | 로그의 주인
**목표:** UPDATE_SCORE 작업을 한 계정의 이름, 변경대상, 시각을 찾으세요.

**힌트:** `A.player_id=L.actor_id / L.action='UPDATE_SCORE'`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_13 = '''
SELECT A.name AS 계정, L.target_id AS 대상번호, L.log_time AS 시각
FROM access_logs L
JOIN players A ON TODO
WHERE TODO;
'''
run_sql(mission_13)

In [ ]:
ANSWERS[13] = "SELECT A.name AS 계정,L.target_id AS 대상번호,L.log_time AS 시각 FROM access_logs L JOIN players A ON A.player_id=L.actor_id WHERE L.action='UPDATE_SCORE';"
check(13, mission_13)

### 🕵️ 미션 14 · 사건 14 | 로그 교차검증
**목표:** 변경 기록과 접속 로그를 시각·대상으로 연결해 변경 전후 점수를 조회하세요.

**힌트:** `H.player_id=L.target_id AND H.changed_at=L.log_time`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_14 = '''
SELECT H.old_points, H.new_points, L.device
FROM score_history H
JOIN access_logs L ON TODO
WHERE L.action = 'UPDATE_SCORE';
'''
run_sql(mission_14)

In [ ]:
ANSWERS[14] = "SELECT H.old_points,H.new_points,L.device FROM score_history H JOIN access_logs L ON H.player_id=L.target_id AND H.changed_at=L.log_time WHERE L.action='UPDATE_SCORE';"
check(14, mission_14)

### 🕵️ 미션 15 · 사건 15 | 최종 사건 보고서
**목표:** 계정명·변경된 선수명·변경전/후 점수를 한 표로 출력하세요.

**힌트:** `H.player_id=L.target_id AND H.changed_at=L.log_time`  결과를 먼저 예상한 뒤 TODO를 바꾸고 실행하세요.

In [ ]:
mission_15 = '''
SELECT A.name AS 계정, P.name AS 대상선수, H.old_points AS 변경전, H.new_points AS 변경후
FROM access_logs L
JOIN players A ON A.player_id=L.actor_id
JOIN players P ON P.player_id=L.target_id
JOIN score_history H ON TODO
WHERE L.action='UPDATE_SCORE';
'''
run_sql(mission_15)

In [ ]:
ANSWERS[15] = "SELECT A.name AS 계정,P.name AS 대상선수,H.old_points AS 변경전,H.new_points AS 변경후 FROM access_logs L JOIN players A ON A.player_id=L.actor_id JOIN players P ON P.player_id=L.target_id JOIN score_history H ON H.player_id=L.target_id AND H.changed_at=L.log_time WHERE L.action='UPDATE_SCORE';"
check(15, mission_15)

## 3. 사건 보고서
**확인할 내용:** (1) 원래 9,800점을 기록한 선수 (2) 14:32에 점수가 어떻게 바뀌었는지 (3) `UPDATE_SCORE`를 실행한 계정 (4) 같은 시각·대상의 점수 변경 이력 (5) 실제 행위자를 단정할 수 없는 이유.

> **결론 작성 예:** "자료에는 ___ 계정이 ___시각에 ___ 선수의 점수를 ___점에서 ___점으로 수정한 작업이 기록되어 있다. 다만 계정 사용자가 실제 당사자인지는 추가 확인이 필요하다."

In [ ]:
progress()

## 4. CRUD 보너스 실습 — 원본 사건 데이터는 건드리지 않아요
`CREATE` 테이블 만들기 → `INSERT` 추가 → `SELECT` 조회 → `UPDATE` 수정 → `DELETE` 삭제.

**실수 방지:** 아래는 별도의 `practice_notes` 테이블만 사용합니다. 원본 `scores` 점수를 바꾸지 마세요.

In [ ]:
conn.execute('DROP TABLE IF EXISTS practice_notes')
conn.execute('CREATE TABLE practice_notes (id INTEGER PRIMARY KEY, memo TEXT NOT NULL)')
conn.execute("INSERT INTO practice_notes VALUES (1, '증거 사진 확인')")
conn.commit()
run_sql('SELECT * FROM practice_notes;')

In [ ]:
# TODO: 아래 문장을 하나씩 실행해 기록이 어떻게 바뀌는지 관찰하세요.
# conn.execute("INSERT INTO practice_notes VALUES (2, '접속 기록 확인')")
# conn.execute("UPDATE practice_notes SET memo='점수 변경 내역 확인' WHERE id=1")
# conn.execute('DELETE FROM practice_notes WHERE id=2')
# conn.commit()
# run_sql('SELECT * FROM practice_notes;')

## 5. 추가 도전 (자유 서술형)
1. 3학년 선수의 이름과 팀을 찾아보세요 (`WHERE`).
2. 팀별 평균 현재 점수를 소수 첫째 자리로 출력하세요 (`JOIN + AVG + GROUP BY`). 점수가 없는 선수는 평균 계산에서 제외됩니다.
3. 로그에서 `PC-07`의 작업을 시간 순으로 표시하세요 (`WHERE + ORDER BY`).
4. `score_history`에서 각 선수의 변경 횟수를 계산하세요 (`GROUP BY`).
5. 점수 기록이 없는 선수까지 이름과 점수를 표시하세요 (`LEFT JOIN`).

```sql
-- 직접 작성할 공간
SELECT ...;
```

In [ ]:
my_sql = '''
-- TODO: 직접 작성한 SQL을 이곳에 넣으세요.
'''
run_sql(my_sql)

## 6. 자기 평가
- [ ] DB · 테이블 · 행 · 열을 구분할 수 있다.
- [ ] SELECT, WHERE, ORDER BY, LIMIT를 이용해 원하는 데이터를 찾을 수 있다.
- [ ] COUNT, GROUP BY, HAVING으로 통계를 작성할 수 있다.
- [ ] 기본키와 연결 열의 관계를 설명할 수 있다.
- [ ] INNER JOIN / LEFT JOIN / IS NULL을 적용할 수 있다.
- [ ] 두 개 이상의 JOIN으로 여러 증거를 연결할 수 있다.
- [ ] 로그의 기록과 실제 사람의 행동을 구분해 해석할 수 있다.

**재도전:** 데이터가 이상하면 준비 셀부터 다시 실행하세요. 학습 파일 저장은 `파일 → 드라이브에 사본 저장`을 활용하세요.

### 참고 문서
- SQLite SELECT: https://www.sqlite.org/lang_select.html
- Python sqlite3: https://docs.python.org/3/library/sqlite3.html
- Google Colab: https://colab.research.google.com/